# Task 1: Video Pre-processing & Keyframe Extraction (SBD)

**Mục tiêu:** So sánh phương pháp trích xuất khung hình hiện tại (Uniform/OpenCV) với SOTA (TransNetV2 / AutoShot).

Notebook này được thiết kế để chạy trên Kaggle GPU T4.

In [1]:
# 1. Cài đặt các thư viện cần thiết
!pip install opencv-python pillow ffmpeg-python

# Tải mô hình TransNetV2 từ GitHub để test
!git clone https://github.com/soCzech/TransNetV2.git
import sys
sys.path.append('/kaggle/working/TransNetV2/inference')

Cloning into 'TransNetV2'...
remote: Enumerating objects: 362, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 362 (delta 71), reused 71 (delta 71), pack-reused 274 (from 1)
Receiving objects: 100% (362/362), 95.25 KiB | 5.60 MiB/s, done.
Resolving deltas: 100% (210/210), done.
Filtering content: 100% (3/3), 34.77 MiB | 6.94 MiB/s, done.


In [2]:
import cv2
import time

# --- P1: Baseline (OpenCV Uniform Sampling như hiện tại) ---
def baseline_extraction(video_path, num_frames=10):
    start_time = time.time()
    cap = cv2.VideoCapture(video_path)
    frames = []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total // num_frames)
    
    for i in range(0, total, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
        if len(frames) == num_frames: break
    cap.release()
    end_time = time.time()
    return frames, end_time - start_time

print('Hàm Baseline đã sẵn sàng.')

Hàm Baseline đã sẵn sàng.


In [3]:
# --- P2: SOTA (TransNetV2 / Scene Boundary Detection) ---
# Bạn tải lên 1 video vào /kaggle/input/ và trỏ đường dẫn tới đây
video_path = '/kaggle/input/datasets/phmthanhhng27/video-l30/video/L30_V001.mp4' # Thay đổi path

try:
    from transnetv2 import TransNetV2
    model = TransNetV2()
    
    start_time = time.time()
    video_frames, single_frame_predictions, all_frame_predictions = model.predict_video(video_path)
    scenes = model.predictions_to_scenes(single_frame_predictions)
    end_time = time.time()
    
    print(f'TransNetV2 tìm thấy {len(scenes)} phân cảnh trong {end_time - start_time:.2f} giây')
except Exception as e:
    print('Vui lòng upload video thật để test TransNetV2:', e)

[TransNetV2] Using weights from /kaggle/working/TransNetV2/inference/transnetv2-weights/.


I0000 00:00:1785738678.915492      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785738678.918275      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[TransNetV2] Extracting frames from /kaggle/input/datasets/phmthanhhng27/video-l30/video/L30_V001.mp4
[TransNetV2] Processing video frames 3162/3162
TransNetV2 tìm thấy 52 phân cảnh trong 14.14 giây


In [ ]:
## Đánh giá:
- So sánh thời gian chạy (Execution Time).
- So sánh độ chính xác: OpenCV cắt ngẫu nhiên có thể dính frame bị mờ hoặc frame chuyển cảnh. SOTA (AutoShot/TransNetV2) sẽ cắt chính xác ranh giới cảnh (scene boundary), giúp các model OCR/VLM phía sau đọc hình ảnh rõ nét hơn.